# 🧩 Mini-Lab: Define the Interface, Keep the Mock

**Module 6: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Locate** where a vendor call leaked into business logic and mark the seam
2. **Write** an `AIClient` Protocol so backends are interchangeable
3. **Make** the existing mock conform and inject it into the pipeline

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Interface seam | Small typed contract; adapter returns your dataclass, never their object |
| Dependency injection | Pass the backend in; callers choose, pipeline code stays unchanged |


---
## Teardown: the repo that died when the API changed


## The Scenario

Imagine you inherited a small codebase that classifies customer support tickets. The original developer hard-coded the OpenAI API call **directly inside** the business logic. One day the vendor changes its API, raises prices, or goes down — and the entire application breaks.

We'll look at this "bad" code, pinpoint exactly where the vendor dependency leaks in, and then mark where the **seam** should go.

## Step 1 — The Tightly-Coupled Code

Below is a simplified ticket classifier. Read through it and notice how many lines **know about the vendor** (OpenAI-specific imports, model names, response shapes).

In [1]:
# ❌ Tightly-coupled code — the vendor is everywhere

COUPLED_CODE = '''
import openai                                       # 🔴 vendor import

def classify_ticket(ticket_text: str) -> str:
    """Classify a support ticket into a category."""

    # --- business logic: build the prompt ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\\n"
        f"Ticket: {ticket_text}\\n"
        f"Category:"
    )

    # --- vendor call buried in the function ---
    client = openai.OpenAI()                        # 🔴 vendor client
    response = client.chat.completions.create(      # 🔴 vendor API shape
        model="gpt-4o-mini",                        # 🔴 vendor model name
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content       # 🔴 vendor response shape

    # --- business logic: validate the answer ---
    category = raw.strip().lower()
    if category not in categories:
        category = "other"
    return category
'''

print(COUPLED_CODE)


import openai                                       # 🔴 vendor import

def classify_ticket(ticket_text: str) -> str:
    """Classify a support ticket into a category."""

    # --- business logic: build the prompt ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\n"
        f"Ticket: {ticket_text}\n"
        f"Category:"
    )

    # --- vendor call buried in the function ---
    client = openai.OpenAI()                        # 🔴 vendor client
    response = client.chat.completions.create(      # 🔴 vendor API shape
        model="gpt-4o-mini",                        # 🔴 vendor model name
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content       # 🔴 vendor response shape

    # --- business logic: validate the answer ---
    category = raw.strip().lower()
    if category not in categories:
        category = "oth

## Step 2 — Counting the Vendor Touch-Points

Let's systematically find every line that is **vendor-specific**. If OpenAI disappeared tomorrow, which lines would break?

In [2]:
# Lines that reference the vendor (OpenAI)
vendor_lines = [
    ("import openai",                        "Vendor import — ties file to openai package"),
    ("client = openai.OpenAI()",             "Vendor client instantiation"),
    ("client.chat.completions.create(...)",  "Vendor-specific API method"),
    ('model="gpt-4o-mini"',                  "Vendor-specific model name"),
    ("response.choices[0].message.content",  "Vendor-specific response shape"),
]

print(f"Vendor touch-points found: {len(vendor_lines)}\n")
for line, reason in vendor_lines:
    print(f"  🔴  {line}")
    print(f"      └─ {reason}\n")

Vendor touch-points found: 5

  🔴  import openai
      └─ Vendor import — ties file to openai package

  🔴  client = openai.OpenAI()
      └─ Vendor client instantiation

  🔴  client.chat.completions.create(...)
      └─ Vendor-specific API method

  🔴  model="gpt-4o-mini"
      └─ Vendor-specific model name

  🔴  response.choices[0].message.content
      └─ Vendor-specific response shape



Five vendor-specific lines are **mixed right in** with the business logic (building the prompt, validating the category). That's the problem: there is no boundary — no **seam** — between *what the app does* and *which AI service it calls*.

## Step 3 — What Breaks When the Vendor Changes?

Let's make this concrete. Suppose the vendor changes its response format (a real thing that happens with API version updates).

In [3]:
# Simulating a vendor API change

# OLD response shape (v1)
old_response = {
    "choices": [{"message": {"content": "billing"}}]
}

# NEW response shape (v2) — vendor renamed the field
new_response = {
    "output": [{"text": "billing"}]
}

# The tightly-coupled code expects the OLD shape
try:
    result = old_response["choices"][0]["message"]["content"]
    print(f"✅ Old format works: '{result}'")
except (KeyError, IndexError) as e:
    print(f"❌ Old format broke: {e}")

try:
    result = new_response["choices"][0]["message"]["content"]
    print(f"✅ New format works: '{result}'")
except (KeyError, IndexError) as e:
    print(f"❌ New format broke: {e}")

print("\n→ The vendor change broke classify_ticket() even though")
print("  the business logic (categories, validation) didn't change.")

✅ Old format works: 'billing'
❌ New format broke: 'choices'

→ The vendor change broke classify_ticket() even though
  the business logic (categories, validation) didn't change.


## Step 4 — Drawing the Seam

An **interface seam** is the line you draw between "business logic" and "vendor call." Everything above the seam is *your application*. Everything below the seam is *the replaceable AI backend*.

Let's label each part of the original function.

In [4]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

# Labeling each section of classify_ticket()

business_lines = [
    "categories = ['billing', 'technical', 'account', 'other']",
    "prompt = f'Classify this support ticket into one of {categories}...'",
    "# (prompt construction is business logic — it knows WHAT to ask)",
    "category = raw.strip().lower()",
    "if category not in categories: category = 'other'",
]

vendor_only_lines = [
    "client = openai.OpenAI()",
    "response = client.chat.completions.create(model='gpt-4o-mini', ...)",
    "raw = response.choices[0].message.content",
]

diagram = """
## Function Anatomy with Seam

```
def classify_ticket(ticket_text):
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • define categories                 │
    │  │  • build the prompt                  │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  VENDOR CODE (replaceable)           │
    │  │  • create OpenAI client              │
    │  │  • call chat.completions.create()    │
    │  │  • dig into response.choices[0]...   │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • validate / clean the answer       │
    │  │  • return the category               │
    │  └─────────────────────────────────────┘
```

The vendor block in the middle is the part you extract behind an interface.
"""

md(diagram)


## Function Anatomy with Seam

```
def classify_ticket(ticket_text):
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • define categories                 │
    │  │  • build the prompt                  │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  VENDOR CODE (replaceable)           │
    │  │  • create OpenAI client              │
    │  │  • call chat.completions.create()    │
    │  │  • dig into response.choices[0]...   │
    │  └─────────────────────────────────────┘
    │
    │  ═══════════ ✂ SEAM HERE ✂ ════════════
    │
    │  ┌─────────────────────────────────────┐
    │  │  BUSINESS LOGIC (yours)              │
    │  │  • validate / clean the answer       │
    │  │  • return the category               │
    │  └─────────────────────────────────────┘
```

The vendor block in the middle is the part you extract behind an interface.


## Step 5 — What the Seam Enables

Once you extract the vendor block behind an interface, the business logic only knows:

> *"I give you a prompt string, you give me back a response string."*

It doesn't care **who** answers — OpenAI, Anthropic, a local Ollama model, or a mock that returns `"billing"` every time for testing.

In [5]:
# What the business logic SHOULD look like after placing the seam

after_seam = """
## After Placing the Seam

```python
def classify_ticket(ticket_text: str, ai_client) -> str:
    # --- business logic only ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\\n"
        f"Ticket: {ticket_text}\\n"
        f"Category:"
    )

    # ONE call through the seam — no vendor details here
    raw = ai_client.complete(prompt)   # ← the interface seam

    # --- business logic continues ---
    category = raw.strip().lower()
    if category not in categories:
        category = "other"
    return category
```

**Zero vendor imports. Zero vendor response parsing.** The function works
with *any* backend that has a `.complete(prompt) -> str` method.
"""

md(after_seam)


## After Placing the Seam

```python
def classify_ticket(ticket_text: str, ai_client) -> str:
    # --- business logic only ---
    categories = ["billing", "technical", "account", "other"]
    prompt = (
        f"Classify this support ticket into one of {categories}.\n"
        f"Ticket: {ticket_text}\n"
        f"Category:"
    )

    # ONE call through the seam — no vendor details here
    raw = ai_client.complete(prompt)   # ← the interface seam

    # --- business logic continues ---
    category = raw.strip().lower()
    if category not in categories:
        category = "other"
    return category
```

**Zero vendor imports. Zero vendor response parsing.** The function works
with *any* backend that has a `.complete(prompt) -> str` method.


## Step 6 — Quick Checklist: Spotting a Missing Seam

Use this checklist on any codebase to detect a missing interface seam.

In [6]:
checklist = [
    ("import openai / import anthropic in business-logic files",
     "Vendor import should only appear in the backend module"),
    ("Model name hard-coded in the function",
     "Model selection belongs in configuration, not logic"),
    ("response.choices[0].message.content deep in your code",
     "Response parsing belongs inside the backend adapter"),
    ("Cannot run tests without a real API key",
     "A seam lets you inject a mock — no key needed"),
    ("Changing provider requires editing many files",
     "With a seam, you only change the backend module"),
]

print("🔍 Missing-Seam Checklist\n")
for i, (symptom, fix) in enumerate(checklist, 1):
    print(f"  {i}. Symptom:  {symptom}")
    print(f"     Fix:      {fix}\n")

🔍 Missing-Seam Checklist

  1. Symptom:  import openai / import anthropic in business-logic files
     Fix:      Vendor import should only appear in the backend module

  2. Symptom:  Model name hard-coded in the function
     Fix:      Model selection belongs in configuration, not logic

  3. Symptom:  response.choices[0].message.content deep in your code
     Fix:      Response parsing belongs inside the backend adapter

  4. Symptom:  Cannot run tests without a real API key
     Fix:      A seam lets you inject a mock — no key needed

  5. Symptom:  Changing provider requires editing many files
     Fix:      With a seam, you only change the backend module



## Recap

In **mini-seam-teardown**, we identified the problem: vendor-specific code (imports, model names, response parsing) was tangled with business logic. We drew a ✂ seam line showing *where* to cut.

Now we'll **build what goes at that seam** — a small typed interface (a Python `Protocol`) and the dependency-injection pattern to wire it up.

## Step 1 — Define the Data Shapes

Before defining the interface, we need to agree on **what goes in** and **what comes out**. Dataclasses give us lightweight, typed containers for this.

In [1]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class AIRequest:
    """What the business logic sends to the AI backend."""
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    """What the AI backend returns to the business logic."""
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


# Quick test: create an instance to confirm the shape
req = AIRequest(prompt="Hello!", temperature=0.5)
print(f"AIRequest  → prompt={req.prompt!r}, temp={req.temperature}, max_tokens={req.max_tokens}")

resp = AIResponse(text="Hi there!", model="mock-1.0")
print(f"AIResponse → text={resp.text!r}, model={resp.model}")

AIRequest  → prompt='Hello!', temp=0.5, max_tokens=256
AIResponse → text='Hi there!', model=mock-1.0


**Why dataclasses?**  
- They are self-documenting: field names and types are visible at a glance.  
- They enforce a consistent shape — every backend receives the same `AIRequest` and returns the same `AIResponse`.  
- Default values (like `temperature=0.0`) keep call sites clean.

## Step 2 — Define the Protocol (The Interface)

A `Protocol` is Python's way of saying *"any class with this method signature is compatible."* No inheritance required — if it has the right method, it fits.

In [2]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class AIClient(Protocol):
    """Any AI backend must implement this single method."""

    def complete(self, request: AIRequest) -> AIResponse:
        """Send a request and get a response."""
        ...


print("AIClient protocol defined.")
print(f"Required method: complete(request: AIRequest) -> AIResponse")

AIClient protocol defined.
Required method: complete(request: AIRequest) -> AIResponse


**Key points:**
- `@runtime_checkable` lets us use `isinstance()` to verify a backend at runtime.
- The protocol has exactly **one method**: `complete`. This is the seam from the previous lab.
- Any class that implements `complete(AIRequest) -> AIResponse` is a valid `AIClient` — no base class needed.

## Step 3 — Build Two Backends

We'll create a **mock** backend (for testing) and a **simple echo** backend. Both satisfy the `AIClient` protocol without inheriting from it.

In [3]:
class MockAIClient:
    """Always returns a fixed answer — perfect for testing."""

    def __init__(self, fixed_answer: str = "billing"):
        self.fixed_answer = fixed_answer

    def complete(self, request: AIRequest) -> AIResponse:
        return AIResponse(text=self.fixed_answer, model="mock-1.0")


class EchoAIClient:
    """Echoes the prompt back — useful for debugging pipelines."""

    def complete(self, request: AIRequest) -> AIResponse:
        return AIResponse(
            text=f"[echo] {request.prompt}",
            model="echo-1.0",
        )


# Verify both satisfy the protocol at runtime
mock = MockAIClient()
echo = EchoAIClient()

print(f"MockAIClient satisfies AIClient? {isinstance(mock, AIClient)}")
print(f"EchoAIClient satisfies AIClient? {isinstance(echo, AIClient)}")

MockAIClient satisfies AIClient? True
EchoAIClient satisfies AIClient? True


Neither class inherits from `AIClient`. Python's `Protocol` uses **structural subtyping** — if the method signature matches, the contract is satisfied. This is often called *duck typing with guardrails*.

## Step 4 — Dependency Injection in Action

Now let's write the ticket classifier from the previous lab — but this time, the AI backend is **injected** as a parameter. The function has zero vendor imports.

In [4]:
def classify_ticket(ticket_text: str, ai_client: AIClient) -> str:
    """Classify a support ticket. The AI backend is injected."""

    # --- business logic: build the request ---
    categories = ["billing", "technical", "account", "other"]
    request = AIRequest(
        prompt=(
            f"Classify this support ticket into one of {categories}.\n"
            f"Ticket: {ticket_text}\nCategory:"
        ),
        temperature=0.0,
    )

    # --- the seam: one call, no vendor details ---
    response = ai_client.complete(request)

    # --- business logic: validate the answer ---
    category = response.text.strip().lower()
    if category not in categories:
        category = "other"
    return category


# Use the mock backend — no API key, no network
result = classify_ticket("I was charged twice this month", mock)
print(f"Mock backend → {result}")

Mock backend → billing


**What just happened:**
- `classify_ticket` has **zero vendor imports**.
- The AI backend is a parameter (`ai_client`) — that's dependency injection.
- Swapping from mock to echo (or to a real OpenAI backend later) requires changing **one argument**, not editing the function.

## Step 5 — What Doesn't Fit the Protocol

Let's verify that a class **without** the right method is correctly rejected.

In [5]:
class BrokenClient:
    """Missing the complete() method entirely."""

    def generate(self, text: str) -> str:
        return text


broken = BrokenClient()
print(f"BrokenClient satisfies AIClient? {isinstance(broken, AIClient)}")
print()
print("→ The protocol catches the mismatch: no complete() method found.")

BrokenClient satisfies AIClient? False

→ The protocol catches the mismatch: no complete() method found.


## Step 6 — The Full Picture

Let's visualize the architecture we just built.

In [6]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

diagram = """
## Architecture After This Lab

```
┌──────────────────────────────────┐
│  Business Logic                  │
│  classify_ticket(text, ai_client)│
│  • builds AIRequest              │
│  • validates AIResponse          │
│  • zero vendor imports           │
└──────────┬───────────────────────┘
           │  ai_client.complete(request)
           │  (the seam)
    ┌──────┴──────┐
    │  AIClient   │  ← Protocol (interface)
    │  Protocol   │
    └──────┬──────┘
     ┌─────┼─────────┐
     │     │         │
     ▼     ▼         ▼
  Mock   Echo   (Future: OpenAI,
                  Ollama, etc.)
```

**Dependency injection** = the caller decides which backend to pass in.  
**Protocol** = the contract that all backends must satisfy.  
**Dataclasses** = the typed shapes that flow across the seam.
"""

md(diagram)


## Architecture After This Lab

```
┌──────────────────────────────────┐
│  Business Logic                  │
│  classify_ticket(text, ai_client)│
│  • builds AIRequest              │
│  • validates AIResponse          │
│  • zero vendor imports           │
└──────────┬───────────────────────┘
           │  ai_client.complete(request)
           │  (the seam)
    ┌──────┴──────┐
    │  AIClient   │  ← Protocol (interface)
    │  Protocol   │
    └──────┬──────┘
     ┌─────┼─────────┐
     │     │         │
     ▼     ▼         ▼
  Mock   Echo   (Future: OpenAI,
                  Ollama, etc.)
```

**Dependency injection** = the caller decides which backend to pass in.  
**Protocol** = the contract that all backends must satisfy.  
**Dataclasses** = the typed shapes that flow across the seam.


## 🎯 Summary

### Key Takeaways

1. **Protocol as interface** — a Python `Protocol` defines the contract (`complete(AIRequest) -> AIResponse`) that any AI backend must satisfy, without requiring inheritance
2. **Dataclass shapes** — `AIRequest` and `AIResponse` dataclasses make the data flowing across the seam explicit, typed, and self-documenting
3. **Dependency injection** — passing the `ai_client` as a parameter means the business logic never imports or knows about any specific vendor
4. **Instant testability** — a `MockAIClient` that returns a fixed answer lets you test the entire pipeline without an API key or network access